# 06. Entrenamiento del Modelo Final

Entrena el recomendador Two Towers definitivo sobre la totalidad de los pares de
co-compra y persiste las torres y los embeddings del catálogo para la fase de servido.

La configuración de hiperparámetros no se fija en este cuaderno: se lee de
`experimentos/hparam_selected.json`, producido por la búsqueda del cuaderno 16. El
número de épocas tampoco es un valor prefijado, sino el que determina una parada
temprana sobre el Recall@100 medido contra el catálogo completo.

**Similitud.** Las dos torres emiten vectores unitarios (normalización L2 a la salida)
y la pérdida usa temperatura. De este modo el producto punto que optimiza el modelo es
exactamente la similitud coseno con la que el índice FAISS recupera candidatos en
servido (cuaderno 11): el objetivo de entrenamiento, la métrica de evaluación y la
función de recuperación son la misma.

**Orden de importación.** TensorFlow debe importarse antes que Polars. Ambos distribuyen
su propia copia de abseil y PyArrow exporta los mismos símbolos; si Arrow se carga
primero, el entrenamiento se bloquea de forma indefinida, sin consumo de CPU y sin error.

In [1]:
import os
_d = os.getcwd()
while not os.path.isdir("data_processed") and os.path.dirname(_d) != _d:
    os.chdir(".."); _d = os.getcwd()
assert os.path.isdir("data_processed"), f"No se encontró data_processed/ desde {os.getcwd()}"
print("Directorio de trabajo:", os.getcwd())

Directorio de trabajo: /Users/allan/Documents/maestria/tesis v4


In [2]:
import tensorflow as tf
import tensorflow_recommenders as tfrs
import numpy as np
import polars as pl
import pandas as pd
import json, time, gc

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

with open("experimentos/hparam_selected.json") as fh:
    HP = json.load(fh)

print("TensorFlow", tf.__version__, "| TFRS", tfrs.__version__, "| Polars", pl.__version__)
print("\nConfiguración seleccionada (cuaderno 16):")
for k, v in HP.items():
    print(f"  {k}: {v}")

TensorFlow 2.21.0 | TFRS v0.7.2 | Polars 1.41.2

Configuración seleccionada (cuaderno 16):
  batch_size: 1024
  remove_accidental_hits: True
  dropout: 0.0
  normalize: True
  temperature: 0.05
  max_epochs: 8
  patience: 2
  seed: 42
  seleccion_frac: 0.25
  metrica_seleccion: Recall@100 coseno
  cos_recall100_seleccion: 0.325
  mejor_epoca_seleccion: 3


## 1. Carga de datos y preprocesamiento del catálogo

Se imputan los nulos del catálogo con las mismas reglas de las fases 2 y 3 y se cruzan
los atributos del producto candidato (`COD_PROD_2`) sobre los pares de entrenamiento.

In [3]:
train_df = pl.read_parquet("data_processed/retrieval_train.parquet")
val_df = pl.read_parquet("data_processed/retrieval_val.parquet")
products_df = pl.read_parquet("data_processed/products_catalog.parquet").unique(subset=["id_producto"])
products_df = products_df.with_columns([
    pl.col("marca").fill_null("SIN_MARCA"),
    pl.col("familia1").fill_null("SIN_CATEGORIA"),
    pl.col("familia2").fill_null("SIN_SUBCATEGORIA"),
    pl.col("precio").fill_null(0.0),
    pl.col("peso_unitario").fill_null(0.0)])

cand_rename = {c: f"cand_{c}" if c != "id_producto" else c for c in products_df.columns}
products_renamed = products_df.rename(cand_rename)

def join_cand(df):
    return df.join(products_renamed, left_on="COD_PROD_2", right_on="id_producto", how="left").rename({
        "cand_marca": "marca_2", "cand_familia1": "familia1_2", "cand_familia2": "familia2_2",
        "cand_precio": "precio_2", "cand_peso_unitario": "peso_unitario_2"}).with_columns([
        pl.col("marca_2").fill_null("SIN_MARCA"),
        pl.col("familia1_2").fill_null("SIN_CATEGORIA"),
        pl.col("familia2_2").fill_null("SIN_SUBCATEGORIA"),
        pl.col("precio_2").fill_null(0.0),
        pl.col("peso_unitario_2").fill_null(0.0)])

print(f"Pares de entrenamiento: {train_df.height:,}")
print(f"Pares de validación:    {val_df.height:,}")
print(f"Catálogo de productos:  {products_df.height:,}")

Pares de entrenamiento: 32,501,962
Pares de validación:    4,331,284
Catálogo de productos:  20,683


## 2. Vocabularios y tabla de corrección LogQ

Los vocabularios se ordenan para que la correspondencia entre cada categoría y su fila
en la matriz de embeddings sea determinista y los pesos guardados puedan recargarse en
cualquier otro cuaderno sin ambigüedad.

La probabilidad de muestreo $P(i)$ de la corrección LogQ se estima sobre la frecuencia
de aparición de cada producto como positivo en el conjunto de entrenamiento completo.

In [4]:
vocab_ruc = sorted(train_df["RUC"].unique().to_list())
vocab_ciudad = sorted(train_df["CIUDAD"].unique().to_list())
vocab_ruta = sorted(train_df["RUTA"].unique().to_list())
vocab_products = sorted(products_df["id_producto"].unique().to_list())
vocab_marca = sorted(products_df["marca"].unique().to_list())
vocab_familia1 = sorted(products_df["familia1"].unique().to_list())
vocab_familia2 = sorted(products_df["familia2"].unique().to_list())

print(f"Clientes (RUC): {len(vocab_ruc):,} | Ciudades: {len(vocab_ciudad)} | Rutas: {len(vocab_ruta)}")
print(f"Productos: {len(vocab_products):,} | Marcas: {len(vocab_marca):,} | "
      f"Familias: {len(vocab_familia1)} / {len(vocab_familia2)}")

total_pairs = train_df.height
counts = train_df.group_by("COD_PROD_2").agg(pl.len().alias("count")).with_columns(
    (pl.col("count") / total_pairs).alias("prob"))
prob_dict = {r["COD_PROD_2"]: r["prob"] for r in counts.iter_rows(named=True)}
logq_table = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        tf.constant(vocab_products, dtype=tf.string),
        tf.constant([prob_dict.get(p, 1e-8) for p in vocab_products], dtype=tf.float32)),
    default_value=1e-8)
del counts, prob_dict; gc.collect()
print("Tabla LogQ inicializada.")

Clientes (RUC): 12,863 | Ciudades: 8 | Rutas: 298
Productos: 20,683 | Marcas: 125 | Familias: 10 / 79


Tabla LogQ inicializada.


## 3. Arquitectura de las torres

Ambas torres proyectan a un espacio común de 128 dimensiones y normalizan su salida en
L2, de modo que el producto punto entre una consulta y un candidato es su similitud coseno.

In [5]:
class QueryTower(tf.keras.Model):
    def __init__(self, vocab_ruc, vocab_ciudad, vocab_ruta, vocab_products,
                 embedding_dim=128, dropout_rate=0.0, normalize=True):
        super().__init__()
        self.normalize = normalize
        self.ruc_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ruc, mask_token=None)
        self.ruc_embedding = tf.keras.layers.Embedding(len(vocab_ruc) + 1, 64, name="ruc_emb")
        self.ciudad_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ciudad, mask_token=None)
        self.ciudad_embedding = tf.keras.layers.Embedding(len(vocab_ciudad) + 1, 16, name="ciudad_emb")
        self.ruta_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ruta, mask_token=None)
        self.ruta_embedding = tf.keras.layers.Embedding(len(vocab_ruta) + 1, 32, name="ruta_emb")
        self.product_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_products, mask_token=None)
        self.product_embedding = tf.keras.layers.Embedding(len(vocab_products) + 1, 64, name="product_emb")
        self.geo_normalization = tf.keras.layers.Normalization(axis=-1)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"), tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embedding_dim, name="query_projection")])

    def call(self, inputs):
        ruc_emb = self.ruc_embedding(self.ruc_lookup(inputs["RUC"]))
        ciudad_emb = self.ciudad_embedding(self.ciudad_lookup(inputs["CIUDAD"]))
        ruta_emb = self.ruta_embedding(self.ruta_lookup(inputs["RUTA"]))
        product_emb = self.product_embedding(self.product_lookup(inputs["COD_PROD"]))
        lat = tf.expand_dims(inputs["LATITUD"], axis=-1)
        lon = tf.expand_dims(inputs["LONGITUD"], axis=-1)
        geo_norm = self.geo_normalization(tf.concat([lat, lon], axis=-1))
        out = self.mlp(tf.concat([ruc_emb, ciudad_emb, ruta_emb, product_emb, geo_norm], axis=-1))
        return tf.math.l2_normalize(out, axis=-1) if self.normalize else out


class CandidateTower(tf.keras.Model):
    def __init__(self, vocab_products, vocab_marca, vocab_familia1, vocab_familia2,
                 embedding_dim=128, dropout_rate=0.0, normalize=True):
        super().__init__()
        self.normalize = normalize
        self.product_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_products, mask_token=None)
        self.product_embedding = tf.keras.layers.Embedding(len(vocab_products) + 1, 64, name="candidate_product_emb")
        self.marca_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_marca, mask_token=None)
        self.marca_embedding = tf.keras.layers.Embedding(len(vocab_marca) + 1, 16, name="candidate_marca_emb")
        self.fam1_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_familia1, mask_token=None)
        self.fam1_embedding = tf.keras.layers.Embedding(len(vocab_familia1) + 1, 16, name="candidate_fam1_emb")
        self.fam2_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_familia2, mask_token=None)
        self.fam2_embedding = tf.keras.layers.Embedding(len(vocab_familia2) + 1, 16, name="candidate_fam2_emb")
        self.continuous_normalization = tf.keras.layers.Normalization(axis=-1)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"), tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embedding_dim, name="candidate_projection")])

    def call(self, inputs):
        prod_emb = self.product_embedding(self.product_lookup(inputs["id_producto"]))
        marca_emb = self.marca_embedding(self.marca_lookup(inputs["marca"]))
        fam1_emb = self.fam1_embedding(self.fam1_lookup(inputs["familia1"]))
        fam2_emb = self.fam2_embedding(self.fam2_lookup(inputs["familia2"]))
        precio = tf.expand_dims(inputs["precio"], axis=-1)
        peso = tf.expand_dims(inputs["peso_unitario"], axis=-1)
        cont_norm = self.continuous_normalization(tf.concat([precio, peso], axis=-1))
        out = self.mlp(tf.concat([prod_emb, marca_emb, fam1_emb, fam2_emb, cont_norm], axis=-1))
        return tf.math.l2_normalize(out, axis=-1) if self.normalize else out

## 4. Modelo acoplado con negativos in-batch y corrección LogQ

`remove_accidental_hits` enmascara los casos en que un producto que es positivo para una
consulta aparece como negativo de otra dentro del mismo lote. En un catálogo con la cabeza
de la distribución tan concentrada como el ferretero, esas colisiones son frecuentes y
penalizan justamente a los productos de mayor rotación.

In [6]:
class HardwareTwoTowers(tfrs.Model):
    def __init__(self, query_tower, candidate_tower, logq_table,
                 remove_accidental_hits, temperature):
        super().__init__()
        self.query_tower = query_tower
        self.candidate_tower = candidate_tower
        self.logq_table = logq_table
        self.remove_accidental_hits = remove_accidental_hits
        self.task = tfrs.tasks.Retrieval(remove_accidental_hits=remove_accidental_hits,
                                         temperature=temperature)

    def compute_loss(self, features, training=False):
        query_embeddings = self.query_tower({k: features[k] for k in
            ["RUC", "CIUDAD", "RUTA", "LATITUD", "LONGITUD", "COD_PROD"]})
        candidate_embeddings = self.candidate_tower({k: features[k] for k in
            ["id_producto", "marca", "familia1", "familia2", "precio", "peso_unitario"]})
        kwargs = {"candidate_sampling_probability": self.logq_table.lookup(features["id_producto"])}
        if self.remove_accidental_hits:
            kwargs["candidate_ids"] = features["id_producto"]
        return self.task(query_embeddings, candidate_embeddings, **kwargs)


def make_tf_dataset(df, batch_size, shuffle=False):
    inputs = {
        "RUC": df["RUC"].to_numpy(), "CIUDAD": df["CIUDAD"].to_numpy(), "RUTA": df["RUTA"].to_numpy(),
        "LATITUD": df["LATITUD"].to_numpy().astype(np.float32),
        "LONGITUD": df["LONGITUD"].to_numpy().astype(np.float32),
        "COD_PROD": df["COD_PROD"].to_numpy(), "id_producto": df["COD_PROD_2"].to_numpy(),
        "marca": df["marca_2"].to_numpy(), "familia1": df["familia1_2"].to_numpy(),
        "familia2": df["familia2_2"].to_numpy(),
        "precio": df["precio_2"].to_numpy().astype(np.float32),
        "peso_unitario": df["peso_unitario_2"].to_numpy().astype(np.float32)}
    ds = tf.data.Dataset.from_tensor_slices(inputs)
    if shuffle:
        ds = ds.shuffle(buffer_size=100_000, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

## 5. Métricas de recuperación y parada temprana

El criterio de parada es el Recall@100 sobre el catálogo completo, no la pérdida in-batch:
esta última depende de qué otros productos compartieron lote con cada ejemplo y no informa
sobre la capacidad de recuperación del modelo.

Se reportan las métricas bajo coseno y bajo producto punto. Con torres normalizadas ambas
coinciden, lo que verifica que la similitud optimizada y la servida son la misma.

In [7]:
val_joined = join_cand(val_df)
val_sample = val_joined.sample(n=min(2000, val_joined.height), seed=SEED)
targets = val_sample["COD_PROD_2"].to_numpy()
catalog_ids = products_df["id_producto"].to_list()
id2c = {p: i for i, p in enumerate(catalog_ids)}
K_LIST = [10, 50, 100]

_cand_inputs = {
    "id_producto": products_df["id_producto"].to_numpy(), "marca": products_df["marca"].to_numpy(),
    "familia1": products_df["familia1"].to_numpy(), "familia2": products_df["familia2"].to_numpy(),
    "precio": products_df["precio"].to_numpy().astype(np.float32),
    "peso_unitario": products_df["peso_unitario"].to_numpy().astype(np.float32)}
_query_inputs = {
    "RUC": val_sample["RUC"].to_numpy(), "CIUDAD": val_sample["CIUDAD"].to_numpy(),
    "RUTA": val_sample["RUTA"].to_numpy(),
    "LATITUD": val_sample["LATITUD"].to_numpy().astype(np.float32),
    "LONGITUD": val_sample["LONGITUD"].to_numpy().astype(np.float32),
    "COD_PROD": val_sample["COD_PROD"].to_numpy()}

def _score(q, c):
    sim = q @ c.T
    rec = {k: 0.0 for k in K_LIST}; mrr = 0.0; n = len(targets)
    for i in range(n):
        j = id2c.get(targets[i])
        if j is None:
            continue
        rank = 1 + int(np.sum(sim[i] > sim[i, j]))
        mrr += 1.0 / rank
        for k in K_LIST:
            if rank <= k:
                rec[k] += 1.0
    return {f"recall{k}": rec[k] / n for k in K_LIST} | {"mrr": mrr / n}

def retrieval_metrics(query_tower, candidate_tower):
    cds = tf.data.Dataset.from_tensor_slices(_cand_inputs).batch(1024)
    ce = tf.concat([candidate_tower(b, training=False) for b in cds], axis=0).numpy()
    qds = tf.data.Dataset.from_tensor_slices(_query_inputs).batch(1024)
    qe = tf.concat([query_tower(b, training=False) for b in qds], axis=0).numpy()
    dot = _score(qe, ce)
    qn = qe / np.maximum(np.linalg.norm(qe, axis=1, keepdims=True), 1e-12)
    cn = ce / np.maximum(np.linalg.norm(ce, axis=1, keepdims=True), 1e-12)
    return dot, _score(qn, cn), ce

class CosineEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, query_tower, candidate_tower, patience):
        super().__init__()
        self.query_tower = query_tower; self.candidate_tower = candidate_tower
        self.patience = patience; self.best = -1.0; self.wait = 0
        self.best_epoch = -1; self.best_weights = None; self.history = []

    def on_epoch_end(self, epoch, logs=None):
        dot, cos, ce = retrieval_metrics(self.query_tower, self.candidate_tower)
        norms = np.linalg.norm(ce, axis=1)
        self.history.append({"epoca": epoch + 1,
                             **{f"cos_{k}": v for k, v in cos.items()},
                             **{f"dot_{k}": v for k, v in dot.items()},
                             "norma_media": float(norms.mean())})
        print(f"  época {epoch+1}: R@100(cos)={cos['recall100']*100:5.2f}%  "
              f"R@100(dot)={dot['recall100']*100:5.2f}%  MRR={cos['mrr']:.4f}", flush=True)
        if cos["recall100"] > self.best:
            self.best = cos["recall100"]; self.best_epoch = epoch + 1; self.wait = 0
            self.best_weights = (self.query_tower.get_weights(), self.candidate_tower.get_weights())
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"  parada temprana; mejor época: {self.best_epoch}", flush=True)
                self.model.stop_training = True

    def restore(self):
        if self.best_weights is not None:
            self.query_tower.set_weights(self.best_weights[0])
            self.candidate_tower.set_weights(self.best_weights[1])

## 6. Entrenamiento sobre la totalidad de los pares

In [8]:
train_joined = join_cand(train_df)
del train_df; gc.collect()
print(f"Entrenando sobre {train_joined.height:,} pares (100 %)")

tf.random.set_seed(SEED)
query_tower = QueryTower(vocab_ruc, vocab_ciudad, vocab_ruta, vocab_products,
                         dropout_rate=HP["dropout"], normalize=HP["normalize"])
candidate_tower = CandidateTower(vocab_products, vocab_marca, vocab_familia1, vocab_familia2,
                                 dropout_rate=HP["dropout"], normalize=HP["normalize"])
query_tower.geo_normalization.adapt(
    train_joined.select(["LATITUD", "LONGITUD"]).to_numpy().astype(np.float32))
candidate_tower.continuous_normalization.adapt(
    products_df.select(["precio", "peso_unitario"]).to_numpy().astype(np.float32))

model = HardwareTwoTowers(query_tower, candidate_tower, logq_table,
                          remove_accidental_hits=HP["remove_accidental_hits"],
                          temperature=HP["temperature"])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

cb = CosineEarlyStopping(query_tower, candidate_tower, patience=HP["patience"])
t0 = time.time()
model.fit(make_tf_dataset(train_joined, HP["batch_size"], shuffle=True),
          epochs=HP["max_epochs"], verbose=0, callbacks=[cb])
cb.restore()
print(f"\nEntrenamiento completado en {(time.time()-t0)/60:.1f} min")

Entrenando sobre 32,501,962 pares (100 %)


  época 1: R@100(cos)=33.05%  R@100(dot)=33.05%  MRR=0.0354


  época 2: R@100(cos)=33.05%  R@100(dot)=33.05%  MRR=0.0376


  época 3: R@100(cos)=32.85%  R@100(dot)=32.85%  MRR=0.0360


  parada temprana; mejor época: 1



Entrenamiento completado en 68.9 min


## 7. Persistencia de torres y embeddings del catálogo

Se guardan los pesos de las dos torres y los embeddings precomputados de los 20.683
productos, que son la entrada del índice vectorial del cuaderno 11.

In [9]:
os.makedirs("models", exist_ok=True)
query_tower.save_weights("models/query_tower.weights.h5")
candidate_tower.save_weights("models/candidate_tower.weights.h5")

cand_ds = tf.data.Dataset.from_tensor_slices(_cand_inputs).batch(1024)
candidate_embeddings = tf.concat([candidate_tower(b, training=False) for b in cand_ds], axis=0).numpy()
np.save("models/candidate_embeddings.npy", candidate_embeddings)
np.save("models/candidate_ids.npy", products_df["id_producto"].to_numpy())
print(f"Embeddings del catálogo: {candidate_embeddings.shape}")
print(f"Norma media de los embeddings: {np.linalg.norm(candidate_embeddings, axis=1).mean():.4f}")

Embeddings del catálogo: (20683, 128)
Norma media de los embeddings: 1.0000


## 8. Resultado del entrenamiento

La coincidencia entre el Recall bajo coseno y bajo producto punto confirma que el modelo
optimiza la misma función de similitud con la que se recuperan los candidatos en servido.

In [10]:
best = cb.history[cb.best_epoch - 1]
resumen = {"pares_entrenamiento": train_joined.height,
           "mejor_epoca": cb.best_epoch, "epocas_corridas": len(cb.history),
           "hiperparametros": HP,
           "recall10": best["cos_recall10"], "recall50": best["cos_recall50"],
           "recall100": best["cos_recall100"], "mrr": best["cos_mrr"],
           "verificacion_dot_recall100": best["dot_recall100"]}
with open("models/final_model_results.json", "w") as fh:
    json.dump(resumen, fh, indent=2, ensure_ascii=False)

pd.DataFrame(cb.history).to_csv("experimentos/final_training_epochs.csv", index=False)

print("Modelo Two Towers + LogQ — entrenamiento final")
for k in K_LIST:
    print(f"  Recall@{k}: {best[f'cos_recall{k}']*100:6.2f}%")
print(f"  MRR:       {best['cos_mrr']:.4f}")
print(f"\nVerificación coseno = producto punto: "
      f"{best['cos_recall100']*100:.2f}% / {best['dot_recall100']*100:.2f}%")

Modelo Two Towers + LogQ — entrenamiento final
  Recall@10:   7.40%
  Recall@50:  21.50%
  Recall@100:  33.05%
  MRR:       0.0354

Verificación coseno = producto punto: 33.05% / 33.05%
